# 日内波动与震荡强度报表

回答："这段时间里，哪些证券日内波动足够大、来回震荡多、扣除成本后有可交易的空间"。
规范：`docs/intraday_volatility_report_spec.md`；计算逻辑在 `trader.analytics`，这里只设参数、看结果。

**用法**：改下面参数单元格里的证券和日期，然后菜单 *Run → Run All Cells*。
只能分析本地已有数据的证券；缺数据的会在摘要里列出，请先用数据管理（`run_downloader.bat`）下载。

In [ ]:
# ===== 参数 =====
from trader.analytics import ReportParams

tickers = [
    "NVDA",
    "AMD",
    "TSLA",
    "AAPL",
    "META",
    "AMZN",
    "GOOG",
    "MSFT",
    "SMH",
    "IWM",
    "DIA",
    "GLD",
    "TLT",
    "XLE",
    "XLF",
    "KORU",
    "NVDL",
    "TSLL",
    "METU",
    "SNDK",
]
start_date = "2026-07-01"
end_date = "2026-09-30"

params = ReportParams(
    benchmark_tickers=("SPY", "QQQ"),  # 参照证券，始终加入计算
    atr_days=21,  # ATR 天数（Wilder）
    swing_thresholds_atr=(0.25, 0.5),  # 摆动阈值，ATR 倍数
    score_k=0.25,  # 打分和时段指标用的阈值
    vwap_cross_min_hold=3,  # VWAP 穿越后至少保持的分钟数
    cost_tiers_bps=(
        (20, 10),
        (100, 5),
        (None, 3),
    ),  # 单边成本：价格 < $20 10bp，< $100 5bp，其余 3bp
    halt_gap_minutes=10,  # 连续无成交超过该分钟数，标记疑似停牌（不剔除）
    exclude_early_close=True,  # 汇总时剔除提前收盘日
    min_dollar_vol=20e6,  # 成交额中位数低于此值标记 low_liquidity
    time_buckets_min=30,  # 时段分析的时间桶（分钟）
    score_weights={"activity": 0.30, "rotation": 0.30, "swing_value": 0.25, "consistency": 0.15},
)
detail_tickers = None  # 单只证券详情；None = 综合得分前 3 名，也可以写 ["NVDA", "AMD"]

In [ ]:
from IPython.display import Markdown, display

from trader.analytics import build_report
from trader.analytics import report as R

rep = build_report(tickers, start_date, end_date, params)
for note in rep.notes:
    display(Markdown(f"> ⚠ {note}"))

## 1. 参数和数据摘要

In [ ]:
display(R.param_summary(rep))
if len(rep.excluded):
    print("未参与汇总的日子（按证券、原因计数）：")
    display(rep.excluded.groupby(["reason", "symbol"]).size().unstack(0).fillna(0).astype(int))
if len(rep.flagged_halts):
    print("疑似停牌（连续无成交超过阈值，只标记、不剔除）：")
    display(rep.flagged_halts)
print("各证券无成交分钟占比（中位数）：")
display(rep.medians[["missing_ratio"]].T.round(4))

## 2. 成本假设

In [ ]:
display(Markdown(R.COST_NOTE))

## 3. 排名总表
分数是本批证券里的相对排名（0–100）；灰底为参照证券，红字为成交额不足。`*_vs_spy` 是相对 SPY 的比值，可以跨批次比较。

In [ ]:
R.styled_ranking(rep)

## 4. 象限散点图

In [ ]:
R.fig_quadrant(rep);

## 5. 分布对比（前 10 名和参照证券）

In [ ]:
R.fig_boxplots(rep);

## 6. 指标分布
第一版不输出"震荡型 / 趋势型"标签。看过这些分布后再定规则（倾向用相对 SPY 的比值）。

In [ ]:
R.fig_distributions(rep);

## 7. 时段热力图

In [ ]:
R.fig_heatmap(rep);

## 8. 单只证券详情
每日指标的时间序列；一个典型交易日（振幅和摆动次数最接近中位数的一天）的 1 分钟价格图，
红线连接识别出的摆动转折点。**请目测转折点是否合理。**换一天看：`R.fig_detail_day(rep, "NVDA", date(2026, 9, 15))`。

In [ ]:
import matplotlib.pyplot as plt

for sym in detail_tickers or rep.detail_tickers():
    display(Markdown(f"### {sym}"))
    for fig in (R.fig_detail_timeseries(rep, sym), R.fig_detail_day(rep, sym)):
        display(fig)
        plt.close(fig)  # 已经显示过，避免单元格结束时再显示一次
    ex = rep.extremes[rep.extremes["symbol"] == sym]
    print("振幅最大 / body_ratio 最高的 3 天（多半是财报或重大消息日）：")
    display(ex.round(3))

## 9. 指标说明

In [ ]:
display(Markdown(R.METRIC_NOTES))

## 10. 导出
CSV（Excel 可以直接打开）和摆动明细 Parquet 写到 `reports/intraday_vol_<开始>_<结束>/`。
要导出整份 HTML 报告：先保存 notebook（Ctrl+S），再把下面的 `export_html` 改成 `True` 运行这个单元格。

In [ ]:
out = rep.export()
print("已导出到", out)

export_html = False
if export_html:
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable,
            "-m",
            "nbconvert",
            "--to",
            "html",
            "--no-input",
            "日内波动报表.ipynb",
            "--output-dir",
            str(out),
            "--output",
            f"report_{rep.start}_{rep.end}",
        ],
        check=True,
    )
    print("HTML 报告：", out / f"report_{rep.start}_{rep.end}.html")